Let’s dive deeper into how **stacking** and **blending** work with a concrete example of a dataset with **1000 rows**. We'll break down the data splits, explain what happens during training and testing, and discuss potential problems in stacking and their solutions.

---

### **Example: Dataset with 1000 Rows**

#### **Dataset Details**
- Total rows: 1000
- Features: $ X $
- Labels: $ y $

#### **Goal**
Train a stacking/blending ensemble to predict $ y $ from $ X $.

---

### **1. Data Splits for Stacking**

In stacking, we use **cross-validation** to generate meta-features. Here’s how the data is divided:

#### **Step 1: Split into Training and Testing Sets**
- Training set: 80% of the data (800 rows)
- Testing set: 20% of the data (200 rows)

The **testing set** is completely held out and never used during training or validation.

#### **Step 2: Cross-Validation on Training Set**
We split the **training set** (800 rows) into $ K $ folds for cross-validation. Let’s assume $ K = 5 $:
- Each fold contains $ \frac{800}{5} = 160 $ rows.
- For each fold:
  - Train base models on $ K-1 $ folds (640 rows).
  - Validate on the remaining fold (160 rows).

This process ensures that every row in the training set is used once as a validation row.

#### **Step 3: Generate Meta-Features**
For each base model:
- Predict on the validation fold using the trained model.
- Collect these predictions across all folds to form a meta-feature matrix.

At the end of this step:
- You have an 800-row meta-feature matrix (one prediction per row for each base model).
- The true labels for these 800 rows are also retained.

#### **Step 4: Train Meta-Model**
- Use the meta-feature matrix (800 rows) and true labels to train the meta-model.

#### **Step 5: Final Prediction**
- To make predictions on the **testing set** (200 rows):
  1. Use the base models to predict on the testing set.
  2. Pass these predictions to the meta-model to get the final output.

---

### **2. Data Splits for Blending**

Blending is simpler than stacking because it uses a **holdout validation set** instead of cross-validation.

#### **Step 1: Split into Training, Validation, and Testing Sets**
- Training set: 60% of the data (600 rows)
- Validation set: 20% of the data (200 rows)
- Testing set: 20% of the data (200 rows)

#### **Step 2: Train Base Models**
- Train the base models on the **training set** (600 rows).

#### **Step 3: Generate Meta-Features**
- Use the base models to predict on the **validation set** (200 rows).
- These predictions form the meta-feature matrix.

#### **Step 4: Train Meta-Model**
- Use the meta-feature matrix (200 rows) and true labels to train the meta-model.

#### **Step 5: Final Prediction**
- To make predictions on the **testing set** (200 rows):
  1. Use the base models to predict on the testing set.
  2. Pass these predictions to the meta-model to get the final output.

---

### **3. Problems in Stacking and Solutions**

Stacking is powerful but comes with challenges. Below are common problems and their solutions:

#### **Problem 1: Overfitting**
- **Cause**: If the meta-model is too complex or if there’s data leakage (e.g., using the same data for training base models and generating meta-features), the ensemble may overfit.
- **Solution**:
  - Always use cross-validation to generate meta-features.
  - Use a simple meta-model (e.g., linear regression) to avoid overfitting.

#### **Problem 2: Computational Cost**
- **Cause**: Training multiple base models and a meta-model can be computationally expensive, especially with large datasets or complex models.
- **Solution**:
  - Use fewer base models or simpler models (e.g., logistic regression instead of deep neural networks).
  - Optimize hyperparameters to reduce training time.

#### **Problem 3: Poor Diversity Among Base Models**
- **Cause**: If the base models are too similar (e.g., all tree-based models), they may capture the same patterns, reducing the benefit of stacking.
- **Solution**:
  - Use diverse base models (e.g., linear models, tree-based models, neural networks).
  - Experiment with different feature subsets or preprocessing techniques for each base model.

#### **Problem 4: Small Validation Set in Blending**
- **Cause**: In blending, the meta-model is trained on a single holdout validation set, which may be too small to generalize well.
- **Solution**:
  - Use stacking instead of blending to leverage cross-validation.
  - Increase the size of the validation set if possible.

#### **Problem 5: Bias in Meta-Model**
- **Cause**: The meta-model may fail to correct errors in the base models if it’s not expressive enough.
- **Solution**:
  - Use a more flexible meta-model (e.g., a neural network or gradient boosting machine).
  - Include raw features alongside meta-features to give the meta-model more context.

---

### **4. Summary Table: Stacking vs. Blending**

| Aspect               | Stacking                                      | Blending                                     |
|----------------------|-----------------------------------------------|----------------------------------------------|
| **Data Splits**      | Training set split into $ K $ folds          | Training, validation, and testing sets       |
| **Meta-Feature Gen** | Out-of-fold predictions via cross-validation | Predictions on a holdout validation set      |
| **Meta-Model Input** | Larger, more robust dataset                  | Smaller, potentially less robust dataset    |
| **Advantages**       | Better generalization, less prone to overfit | Simpler, faster                            |
| **Disadvantages**    | Computationally expensive                    | Risk of overfitting due to smaller dataset   |

---

### **5. Example Walkthrough with Numbers**

#### **Stacking**
1. Training set: 800 rows → Split into 5 folds (160 rows each).
2. Train base models on 4 folds (640 rows) and validate on 1 fold (160 rows).
3. Repeat for all folds → Generate 800 meta-features.
4. Train meta-model on 800 meta-features.
5. Test on 200 rows → Use base models + meta-model for final predictions.

#### **Blending**
1. Training set: 600 rows → Train base models.
2. Validation set: 200 rows → Generate meta-features.
3. Train meta-model on 200 meta-features.
4. Test on 200 rows → Use base models + meta-model for final predictions.

---

### **Final Answer**
$$
\boxed{\text{Stacking divides data into cross-validation folds for meta-feature generation, while blending uses a holdout validation set. Both methods test on a completely unseen dataset.}}
$$

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Generate synthetic data
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
from sklearn.model_selection import KFold

# Define base models
base_models = [
    ('lr', LogisticRegression(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42))
]

# Meta-model
meta_model = LogisticRegression(random_state=42)

In [3]:
# Initialize meta-feature matrix
meta_features = np.zeros((X_train.shape[0], len(base_models)))

# Perform cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for i, (name, model) in enumerate(base_models):
    print(f"Training base model: {name}")
    for train_idx, val_idx in kf.split(X_train):
        # Split data
        X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]

        # Train base model
        model.fit(X_train_fold, y_train_fold)

        # Predict on validation fold
        meta_features[val_idx, i] = model.predict_proba(X_val_fold)[:, 1]

Training base model: lr
Training base model: rf
Training base model: gb


In [5]:
# Train meta-model
meta_model.fit(meta_features, y_train)

LogisticRegression(random_state=42)

In [6]:
# Generate base model predictions for test set
test_meta_features = np.zeros((X_test.shape[0], len(base_models)))
for i, (name, model) in enumerate(base_models):
    test_meta_features[:, i] = model.predict_proba(X_test)[:, 1]

# Final prediction using meta-model
y_pred = meta_model.predict(test_meta_features)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Stacking Accuracy: {accuracy:.4f}")

Stacking Accuracy: 0.8950


##Blending

In [7]:
# Split training set into training and validation sets
X_train_blend, X_val_blend, y_train_blend, y_val_blend = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42
)

In [8]:
# Train base models
for name, model in base_models:
    print(f"Training base model: {name}")
    model.fit(X_train_blend, y_train_blend)

Training base model: lr
Training base model: rf
Training base model: gb


In [9]:
# Generate meta-features for validation set
val_meta_features = np.zeros((X_val_blend.shape[0], len(base_models)))
for i, (name, model) in enumerate(base_models):
    val_meta_features[:, i] = model.predict_proba(X_val_blend)[:, 1]

In [10]:
# Train meta-model
meta_model.fit(val_meta_features, y_val_blend)

LogisticRegression(random_state=42)

In [11]:
# Generate base model predictions for test set
test_meta_features = np.zeros((X_test.shape[0], len(base_models)))
for i, (name, model) in enumerate(base_models):
    test_meta_features[:, i] = model.predict_proba(X_test)[:, 1]

# Final prediction using meta-model
y_pred = meta_model.predict(test_meta_features)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Blending Accuracy: {accuracy:.4f}")

Blending Accuracy: 0.8750




---

### **1. What is Stacking? How does it differ from Blending?**

**Answer:**
- **Stacking** is an ensemble learning technique where multiple base models (level-0 models) are trained, and their predictions are combined using a meta-model (level-1 model). The meta-model learns how to optimally combine the predictions of the base models. Stacking uses cross-validation to generate out-of-fold predictions for training the meta-model.
- **Blending** is a simpler version of stacking. Instead of cross-validation, blending uses a holdout validation set to generate predictions for the meta-model.

**Key Differences:**
| Aspect               | Stacking                                      | Blending                                     |
|----------------------|-----------------------------------------------|----------------------------------------------|
| **Data Splits**      | Training set split into $ K $ folds          | Training, validation, and testing sets       |
| **Meta-Feature Gen** | Out-of-fold predictions via cross-validation | Predictions on a holdout validation set      |
| **Advantages**       | Better generalization, less prone to overfit | Simpler, faster                            |
| **Disadvantages**    | Computationally expensive                    | Risk of overfitting due to smaller dataset   |

---

### **2. Why is Cross-Validation Important in Stacking?**

**Answer:**
Cross-validation is crucial in stacking because it ensures that the meta-model is trained on **out-of-fold predictions**. This prevents data leakage, where the meta-model might inadvertently learn patterns from the same data used to train the base models. Using cross-validation ensures that the meta-model sees predictions from unseen data, leading to better generalization.

---

### **3. What Are Some Common Base Models Used in Stacking?**

**Answer:**
Common base models in stacking include:
- **Linear Models**: Logistic Regression, Ridge Regression
- **Tree-Based Models**: Random Forests, Gradient Boosting Machines (e.g., XGBoost, LightGBM)
- **Neural Networks**: Simple feedforward networks
- **K-Nearest Neighbors (KNN)**

The key idea is to use diverse models that capture different patterns in the data.

---

### **4. What Are Some Common Meta-Models Used in Stacking?**

**Answer:**
Common meta-models include:
- **Linear Models**: Logistic Regression, Linear Regression
- **Tree-Based Models**: Random Forests, Gradient Boosting Machines
- **Neural Networks**: Simple neural networks for non-linear combinations

The choice of meta-model depends on the complexity of the problem. A simple linear meta-model is often sufficient, but more complex models can be used if the relationships between base model predictions are non-linear.

---

### **5. How Do You Prevent Overfitting in Stacking?**

**Answer:**
To prevent overfitting in stacking:
1. **Use Cross-Validation**: Always generate meta-features using cross-validation to ensure the meta-model is trained on out-of-fold predictions.
2. **Keep the Meta-Model Simple**: Start with a simple meta-model like logistic regression before experimenting with complex models.
3. **Diversify Base Models**: Use a variety of base models to reduce correlation between their predictions.
4. **Regularization**: Apply regularization techniques (e.g., L1/L2 regularization) to both base models and the meta-model.
5. **Holdout Validation**: Reserve a separate validation set to evaluate the performance of the stacked model.

---

### **6. What Are the Advantages of Stacking Over Other Ensemble Methods?**

**Answer:**
Advantages of stacking include:
1. **Improved Accuracy**: By combining diverse models, stacking can achieve better predictive performance than individual models.
2. **Flexibility**: Any combination of base models and meta-models can be used.
3. **Bias-Variance Tradeoff**: Stacking balances bias and variance by leveraging the strengths of different models.
4. **Customizable**: You can include raw features alongside meta-features to give the meta-model more context.

---

### **7. What Are the Limitations of Stacking?**

**Answer:**
Limitations of stacking include:
1. **Computational Cost**: Training multiple base models and a meta-model can be computationally expensive.
2. **Complexity**: Stacking requires careful tuning and can be harder to implement compared to simpler ensemble methods like bagging or boosting.
3. **Overfitting Risk**: If not implemented properly (e.g., without cross-validation), stacking can lead to overfitting.
4. **Interpretability**: Stacked models are often less interpretable than individual models.

---

### **8. When Would You Prefer Blending Over Stacking?**

**Answer:**
Blending is preferred over stacking in the following scenarios:
1. **Small Datasets**: When the dataset is small, cross-validation in stacking may lead to high variance. Blending uses a single holdout set, which is simpler.
2. **Speed**: Blending is computationally faster since it avoids the overhead of cross-validation.
3. **Simplicity**: Blending is easier to implement and debug compared to stacking.

However, blending is generally less robust than stacking due to its reliance on a single validation set.

---

### **9. How Would You Implement Stacking in Practice?**

**Answer:**
Steps to implement stacking:
1. **Split the Data**: Divide the dataset into training and testing sets.
2. **Train Base Models**: Train multiple base models on the training set using cross-validation.
3. **Generate Meta-Features**: Use out-of-fold predictions from the base models to create a meta-feature matrix.
4. **Train Meta-Model**: Train a meta-model on the meta-feature matrix.
5. **Make Predictions**: For new data, use the base models to generate predictions and pass them to the meta-model for the final output.

Example code snippet (using scikit-learn):
```python
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Define base models
base_models = [
    ('lr', LogisticRegression()),
    ('rf', RandomForestClassifier()),
    ('gb', GradientBoostingClassifier())
]

# Meta-model
meta_model = LogisticRegression()

# Generate meta-features using cross-validation
kf = KFold(n_splits=5)
meta_features = np.zeros((X_train.shape[0], len(base_models)))
for i, (name, model) in enumerate(base_models):
    for train_idx, val_idx in kf.split(X_train):
        X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
        model.fit(X_train_fold, y_train_fold)
        meta_features[val_idx, i] = model.predict_proba(X_val_fold)[:, 1]

# Train meta-model
meta_model.fit(meta_features, y_train)
```

---

### **10. Can Stacking Be Applied to Regression Problems?**

**Answer:**
Yes, stacking can be applied to regression problems. The process is similar to classification:
1. Train base models on the training data.
2. Generate meta-features using cross-validation (e.g., predicted values instead of probabilities).
3. Train a meta-model (e.g., linear regression) on the meta-features.
4. Make predictions on new data using the base models and meta-model.

---

### **11. What Is the Role of the Meta-Model in Stacking?**

**Answer:**
The meta-model acts as a "second-level" model that learns how to combine the predictions of the base models. Its role is to:
1. Identify the strengths and weaknesses of each base model.
2. Correct errors made by the base models.
3. Optimize the combination of predictions to improve overall performance.

For example, if one base model performs well on certain subsets of the data and poorly on others, the meta-model can assign higher weights to the reliable predictions.

---

### **12. How Do You Evaluate the Performance of a Stacked Model?**

**Answer:**
To evaluate a stacked model:
1. Use a **separate test set** that was not used during training or validation.
2. Compare metrics such as accuracy, precision, recall, F1-score (for classification) or RMSE, MAE (for regression) on the test set.
3. Perform **cross-validation** on the entire pipeline (base models + meta-model) to get a more robust estimate of performance.

---

### **13. Can You Combine Stacking with Other Techniques Like Feature Engineering?**

**Answer:**
Yes, stacking can be combined with feature engineering. For example:
1. Include **raw features** alongside meta-features to give the meta-model more context.
2. Use feature selection techniques to reduce dimensionality before feeding data into base models.
3. Apply transformations (e.g., scaling, encoding) to preprocess the data for both base models and the meta-model.

This hybrid approach often leads to better performance.

---

### **Final Thoughts**

Stacking and blending are powerful ensemble techniques that require a solid understanding of machine learning fundamentals. By preparing for these interview questions, you’ll demonstrate both theoretical knowledge and practical implementation skills.



##Multi Layered Stacking

### **Multi-Layered Stacking: A Deep Dive**

**Multi-layered stacking** (also known as **stacked generalization with multiple levels**) is an advanced extension of the traditional stacking technique. Instead of having just one meta-model (level-1 model), multi-layered stacking introduces additional layers of meta-models to further refine the predictions.

This approach can be thought of as a "deep ensemble" where each layer learns from the predictions of the previous layer. The idea is to iteratively combine models at different levels, allowing the system to capture increasingly complex relationships in the data.

---

### **1. Core Idea of Multi-Layered Stacking**

The basic architecture of multi-layered stacking looks like this:

1. **Level-0 Models**: Base models trained on the original dataset.
2. **Level-1 Model**: Meta-model trained on the predictions of Level-0 models.
3. **Level-2 Model**: Another meta-model trained on the predictions of Level-1 models.
4. **... and so on**: You can continue adding layers as needed.

Each layer refines the predictions from the previous layer, potentially improving the overall performance.

---

### **2. How It Works**

#### **Step 1: Train Level-0 Models**
- Train multiple base models (e.g., logistic regression, random forest, gradient boosting) on the training data using cross-validation.
- Generate out-of-fold predictions for each base model to create a meta-feature matrix.

#### **Step 2: Train Level-1 Model**
- Use the meta-feature matrix from Level-0 models to train a meta-model (Level-1).
- This meta-model learns how to optimally combine the predictions of the Level-0 models.

#### **Step 3: Train Level-2 Model**
- Treat the predictions of the Level-1 model as inputs to train another meta-model (Level-2).
- Optionally, you can include raw features or other engineered features alongside the predictions.

#### **Step 4: Repeat for Additional Layers**
- Continue adding layers if necessary, where each layer learns from the outputs of the previous layer.

#### **Step 5: Final Prediction**
- For new data:
  1. Use the Level-0 models to generate predictions.
  2. Pass these predictions to the Level-1 model.
  3. Pass the output of the Level-1 model to the Level-2 model, and so on.
  4. The final output comes from the last meta-model.

---

### **3. Example Architecture**

Let’s break down the architecture with an example:

#### **Dataset**
- Features: $ X $
- Labels: $ y $

#### **Level-0 Models**
- Logistic Regression
- Random Forest
- Gradient Boosting

#### **Level-1 Model**
- Logistic Regression (trained on the predictions of Level-0 models)

#### **Level-2 Model**
- Neural Network (trained on the predictions of the Level-1 model)

#### **Final Output**
- The neural network produces the final prediction.

---

### **4. Mathematical Framework**

#### **Level-0 Models**
For $ M $ base models:
$$
\hat{y}_i = f_i(X), \quad i = 1, 2, \dots, M
$$
Each base model generates predictions $ \hat{y}_i $.

#### **Level-1 Model**
The Level-1 model $ g_1 $ takes the predictions of Level-0 models as input:
$$
g_1(\hat{y}_1, \hat{y}_2, \dots, \hat{y}_M)
$$

#### **Level-2 Model**
The Level-2 model $ g_2 $ takes the predictions of the Level-1 model as input:
$$
g_2(g_1(\hat{y}_1, \hat{y}_2, \dots, \hat{y}_M))
$$

#### **Final Prediction**
The final prediction is given by:
$$
\hat{y}_{\text{final}} = g_n(\dots g_2(g_1(\hat{y}_1, \hat{y}_2, \dots, \hat{y}_M)) \dots)
$$

---

### **5. Implementation Example**

Below is a Python implementation of multi-layered stacking using `scikit-learn`:

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Generate synthetic data
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define Level-0 models
level0_models = [
    ('lr', LogisticRegression(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42))
]

# Define Level-1 and Level-2 models
level1_model = LogisticRegression(random_state=42)
level2_model = MLPClassifier(hidden_layer_sizes=(10,), random_state=42)

# Step 1: Train Level-0 models and generate meta-features
kf = KFold(n_splits=5, shuffle=True, random_state=42)
meta_features_level0 = np.zeros((X_train.shape[0], len(level0_models)))

for i, (name, model) in enumerate(level0_models):
    print(f"Training Level-0 model: {name}")
    for train_idx, val_idx in kf.split(X_train):
        X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
        model.fit(X_train_fold, y_train_fold)
        meta_features_level0[val_idx, i] = model.predict_proba(X_val_fold)[:, 1]

# Step 2: Train Level-1 model
level1_model.fit(meta_features_level0, y_train)

# Generate meta-features for Level-2
meta_features_level1 = level1_model.predict_proba(meta_features_level0)[:, 1].reshape(-1, 1)

# Step 3: Train Level-2 model
level2_model.fit(meta_features_level1, y_train)

# Step 4: Make predictions on test set
test_meta_features_level0 = np.zeros((X_test.shape[0], len(level0_models)))
for i, (name, model) in enumerate(level0_models):
    test_meta_features_level0[:, i] = model.predict_proba(X_test)[:, 1]

test_meta_features_level1 = level1_model.predict_proba(test_meta_features_level0)[:, 1].reshape(-1, 1)
y_pred = level2_model.predict(test_meta_features_level1)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Multi-Layered Stacking Accuracy: {accuracy:.4f}")
```

---

### **6. Advantages of Multi-Layered Stacking**

1. **Improved Performance**: By adding more layers, the model can capture increasingly complex patterns in the data.
2. **Flexibility**: You can experiment with different combinations of models at each layer.
3. **Better Generalization**: Each layer refines the predictions, reducing overfitting.

---

### **7. Challenges of Multi-Layered Stacking**

1. **Computational Cost**: Training multiple layers increases the computational overhead.
2. **Overfitting Risk**: Adding too many layers can lead to overfitting, especially if the dataset is small.
3. **Complexity**: Debugging and tuning a multi-layered stack can be challenging.

---

### **8. Practical Tips**

1. **Start Simple**: Begin with a single meta-model and add layers incrementally.
2. **Monitor Overfitting**: Use cross-validation and holdout validation sets to evaluate performance at each layer.
3. **Diversify Models**: Use different types of models at each layer to capture diverse patterns.
4. **Regularization**: Apply regularization techniques to prevent overfitting.

---

### **9. When to Use Multi-Layered Stacking?**

Use multi-layered stacking when:
1. The dataset is large enough to support multiple layers of training.
2. Traditional stacking does not provide sufficient performance gains.
3. You want to experiment with deep ensembles to capture complex relationships.

---

### **Final Answer**
$$
\boxed{\text{Multi-layered stacking extends traditional stacking by adding multiple meta-model layers, enabling deeper refinement of predictions.}}
$$

##Differences

### **Stacking vs Other Ensemble Methods (Minimal Differences)**

1. **Bagging**:
   - Bagging trains multiple models independently on bootstrapped subsets and averages their predictions.
   - Stacking uses a meta-model to learn how to combine base model predictions.

2. **Boosting**:
   - Boosting trains models sequentially, focusing on correcting errors of previous models.
   - Stacking trains models independently and combines them via a meta-model.

3. **Blending**:
   - Blending uses a single holdout validation set for meta-features.
   - Stacking uses cross-validation to generate meta-features, reducing overfitting risk.

4. **Simple Averaging/Voting**:
   - Simple averaging/voting combines predictions directly (e.g., mean or majority vote).
   - Stacking learns an optimal combination using a meta-model.

$$
\boxed{\text{Stacking = Meta-model learns to combine; Others = Fixed rules or sequential corrections.}}
$$